# Email Security & Phishing Detection

**Section 1 of the ML in Cyber Security assignment (25 marks)**

This notebook builds and compares two models for detecting phishing/malicious emails from their text content:

1. **Classic Machine Learning:** TF-IDF features + Logistic Regression
2. **Deep Learning:** Tokenized sequences + an LSTM neural network

**Dataset:** [Enron Spam Data](https://www.kaggle.com/datasets/marcelwiechmann/enron-spam-data) (Kaggle). The dataset's `spam` / `ham` labels are used as a proxy for `phishing` / `legitimate`, consistent with the assignment brief's own suggested dataset list (Enron Spam, SpamAssassin, UCI Phishing). This is a documented limitation discussed at the end of the notebook: not every spam email is a phishing email, but the two problems share the same text-classification setup, and spam corpora are the standard stand-in used in coursework and much of the phishing-detection literature.

See `README.md` in this folder for setup and run instructions, including what to do if you don't have Kaggle API credentials configured.

In [1]:
import os
import re
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc,
)

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = "outputs"
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
MODELS_DIR = os.path.join(OUTPUT_DIR, "models")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("punkt", quiet=True)

print("Setup complete.")

Setup complete.


## 1. Data Acquisition

The cell below tries to download the Enron Spam Data dataset directly from Kaggle via `kagglehub`. This requires Kaggle API credentials to be configured (see `README.md`).

If that fails (no credentials / no internet access), it falls back to reading a manually downloaded copy from `data/enron_spam_data.csv`.

In [2]:
DATA_LOCAL_PATH = os.path.join("data", "enron_spam_data.csv")


def load_enron_spam_data():
    try:
        import kagglehub
        from kagglehub import KaggleDatasetAdapter

        print("Attempting to download 'Enron Spam Data' from Kaggle via kagglehub ...")
        dataset = kagglehub.load_dataset(
            KaggleDatasetAdapter.PANDAS,
            "marcelwiechmann/enron-spam-data",
            "",
        )
        print("Loaded dataset via kagglehub.")
        return dataset
    except Exception as exc:
        print(f"kagglehub download failed: {exc}")
        if os.path.exists(DATA_LOCAL_PATH):
            print(f"Falling back to local file: {DATA_LOCAL_PATH}")
            return pd.read_csv(DATA_LOCAL_PATH)
        raise FileNotFoundError(
            "Could not load the Enron Spam Data dataset.\n"
            "Either:\n"
            "  1) Configure Kaggle API credentials (~/.kaggle/kaggle.json or "
            "KAGGLE_USERNAME/KAGGLE_KEY env vars) so kagglehub can download it, or\n"
            "  2) Manually download the dataset from Kaggle "
            "(marcelwiechmann/enron-spam-data) and place the CSV at "
            f"'{DATA_LOCAL_PATH}'.\n"
            "See README.md for details."
        ) from exc


df = load_enron_spam_data()
print("Shape:", df.shape)
df.head()

Attempting to download 'Enron Spam Data' from Kaggle via kagglehub ...
kagglehub download failed: Unsupported file extension: ''. Supported file extensions are: .csv, .tsv, .json, .jsonl, .xml, .parquet, .feather, .sqlite, .sqlite3, .db, .db3, .s3db, .dl3, .xls, .xlsx, .xlsm, .xlsb, .odf, .ods, .odt


FileNotFoundError: Could not load the Enron Spam Data dataset.
Either:
  1) Configure Kaggle API credentials (~/.kaggle/kaggle.json or KAGGLE_USERNAME/KAGGLE_KEY env vars) so kagglehub can download it, or
  2) Manually download the dataset from Kaggle (marcelwiechmann/enron-spam-data) and place the CSV at 'data/enron_spam_data.csv'.
See README.md for details.

## 2. Column Detection & Label Normalization

The exact column names in the Kaggle mirror of this dataset can vary between versions, so the cell below auto-detects the email-text column and the label column by keyword matching, then normalizes labels to `0` (legitimate/ham) and `1` (phishing/spam).

If auto-detection picks the wrong columns for your downloaded copy, override `TEXT_COL` / `LABEL_COL` manually using the commented-out lines, after checking `df.columns` above.

In [ ]:
def detect_column(columns, keywords):
    lowered = {c: str(c).lower() for c in columns}
    for keyword in keywords:
        for col, low in lowered.items():
            if keyword in low:
                return col
    return None


TEXT_COL = detect_column(df.columns, ["message", "text", "body", "content", "email"])
LABEL_COL = detect_column(df.columns, ["label", "class", "spam/ham", "spam", "ham", "target"])

# Override manually here if auto-detection picks the wrong column, e.g.:
# TEXT_COL = "Message"
# LABEL_COL = "Spam/Ham"

assert TEXT_COL is not None, f"Could not auto-detect a text column among {list(df.columns)}. Set TEXT_COL manually."
assert LABEL_COL is not None, f"Could not auto-detect a label column among {list(df.columns)}. Set LABEL_COL manually."

print(f"Using text column:  '{TEXT_COL}'")
print(f"Using label column: '{LABEL_COL}'")

df = df[[TEXT_COL, LABEL_COL]].dropna().rename(columns={TEXT_COL: "text", LABEL_COL: "label_raw"})


def normalize_label(value):
    if isinstance(value, (int, np.integer, float, np.floating)):
        return int(value)
    text_value = str(value).strip().lower()
    if text_value in ("spam", "phishing", "1", "malicious", "bad"):
        return 1
    if text_value in ("ham", "legitimate", "0", "safe", "good"):
        return 0
    raise ValueError(f"Unrecognized label value: {value!r}")


df["label"] = df["label_raw"].apply(normalize_label)
df = df.drop(columns=["label_raw"]).reset_index(drop=True)

print("\nLabel distribution (0 = legitimate, 1 = phishing/spam):")
print(df["label"].value_counts())

## 3. Remove Duplicate Emails (Prevent Train/Test Leakage)

The raw Enron Spam Data CSV contains a large number of exact-duplicate emails (automated log messages, bulk spam, and forwarded chains that were saved as separate files in the original per-message corpus — over half the rows are duplicates of just a few hundred unique messages). Splitting into train/test on raw rows without deduplication lets identical emails land in both sets, so a model can "cheat" by memorizing an email during training and then matching its exact duplicate at test time — inflating every metric (often to near-100%) without the model actually learning to generalize.

The cell below removes duplicate emails (same `text`), keeping only the first occurrence of each. This must happen **before** the train/test split so no duplicate can end up on both sides of it. A handful of duplicate messages also carried conflicting spam/ham labels in the raw data; keeping only the first occurrence resolves this deterministically.

In [ ]:
n_before = len(df)
df = df.drop_duplicates(subset="text").reset_index(drop=True)
n_after = len(df)

print(f"Removed {n_before - n_after} duplicate emails ({n_before} -> {n_after} rows).")
print("\nLabel distribution after deduplication (0 = legitimate, 1 = phishing/spam):")
print(df["label"].value_counts())

## 4. Exploratory Data Analysis

A quick look at class balance, email length, and a couple of raw samples before any cleaning.

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(x="label", data=df)
plt.xticks([0, 1], ["Legitimate (0)", "Phishing/Spam (1)"])
plt.title("Class Distribution")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "class_distribution.png"), dpi=150)
plt.show()

df["text_length"] = df["text"].astype(str).apply(len)

plt.figure(figsize=(6, 4))
sns.histplot(data=df, x="text_length", hue="label", bins=50, log_scale=(False, True))
plt.title("Email Length Distribution by Class")
plt.xlabel("Character length")
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "text_length_distribution.png"), dpi=150)
plt.show()

print("Sample legitimate email:\n", df[df.label == 0]["text"].iloc[0][:500])
print("\nSample phishing/spam email:\n", df[df.label == 1]["text"].iloc[0][:500])

## 5. Text Preprocessing

Each email is cleaned with the following steps before being fed to either model:

1. Lowercase the text.
2. Strip URLs and HTML tags.
3. Remove punctuation/digits, keeping only alphabetic tokens.
4. Remove English stopwords.
5. Lemmatize remaining tokens (reduce to dictionary root form) and drop very short tokens (length &le; 2).

The result is stored in a new `clean_text` column, which both the TF-IDF/Logistic Regression model and the LSTM's tokenizer are built on.

In [ ]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_RE = re.compile(r"<.*?>")
NON_ALPHA_RE = re.compile(r"[^a-z\s]")


def clean_text(text):
    text = str(text).lower()
    text = URL_RE.sub(" ", text)
    text = HTML_RE.sub(" ", text)
    text = NON_ALPHA_RE.sub(" ", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(tok) for tok in tokens if tok not in stop_words and len(tok) > 2]
    return " ".join(tokens)


df["clean_text"] = df["text"].apply(clean_text)
df = df[df["clean_text"].str.strip().astype(bool)].reset_index(drop=True)

print(f"Rows after cleaning: {len(df)}")
df[["text", "clean_text", "label"]].head()

## 6. Train / Test Split

An 80/20 stratified split is used so both classes keep the same proportion in train and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=SEED, stratify=df["label"]
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 7. Classic Machine Learning — TF-IDF + Logistic Regression

`TfidfVectorizer` converts each cleaned email into a weighted bag-of-unigram/bigram vector (capped at 10,000 features), which a `LogisticRegression` classifier is trained on.

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, random_state=SEED)
log_reg.fit(X_train_tfidf, y_train)

y_pred_lr = log_reg.predict(X_test_tfidf)
y_proba_lr = log_reg.predict_proba(X_test_tfidf)[:, 1]

print("=== Logistic Regression (TF-IDF) ===")
print(classification_report(y_test, y_pred_lr, target_names=["Legitimate", "Phishing/Spam"]))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Legitimate", "Phishing/Spam"],
        yticklabels=["Legitimate", "Phishing/Spam"],
    )
    plt.title(title)
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=150)
    plt.show()


def plot_roc_curve(y_true, y_proba, title, filename):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="grey")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=150)
    plt.show()
    return roc_auc


plot_confusion_matrix(y_test, y_pred_lr, "Logistic Regression - Confusion Matrix", "lr_confusion_matrix.png")
auc_lr = plot_roc_curve(y_test, y_proba_lr, "Logistic Regression - ROC Curve", "lr_roc_curve.png")

joblib.dump(tfidf, os.path.join(MODELS_DIR, "tfidf_vectorizer.joblib"))
joblib.dump(log_reg, os.path.join(MODELS_DIR, "logistic_regression.joblib"))
print("Saved TF-IDF vectorizer and Logistic Regression model to", MODELS_DIR)

## 8. Deep Learning — LSTM

Text is tokenized into integer sequences (vocabulary capped at 10,000 words), padded/truncated to a fixed length of 200 tokens, and fed into an `Embedding -> LSTM -> Dense` network trained end-to-end for binary classification.

In [ ]:
MAX_VOCAB = 10000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_VOCAB, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

lstm_model = Sequential([
    Embedding(input_dim=MAX_VOCAB, output_dim=128, input_length=MAX_LEN),
    LSTM(64, dropout=0.2, recurrent_dropout=0.2),
    Dense(32, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

lstm_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
lstm_model.summary()

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = lstm_model.fit(
    X_train_pad, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1,
)

In [ ]:
def plot_training_history(history, filename):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(history.history["loss"], label="Train Loss")
    axes[0].plot(history.history["val_loss"], label="Val Loss")
    axes[0].set_title("LSTM Loss over Epochs")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="Train Accuracy")
    axes[1].plot(history.history["val_accuracy"], label="Val Accuracy")
    axes[1].set_title("LSTM Accuracy over Epochs")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, filename), dpi=150)
    plt.show()


plot_training_history(history, "lstm_training_curves.png")

y_proba_lstm = lstm_model.predict(X_test_pad).ravel()
y_pred_lstm = (y_proba_lstm >= 0.5).astype(int)

print("=== LSTM ===")
print(classification_report(y_test, y_pred_lstm, target_names=["Legitimate", "Phishing/Spam"]))

plot_confusion_matrix(y_test, y_pred_lstm, "LSTM - Confusion Matrix", "lstm_confusion_matrix.png")
auc_lstm = plot_roc_curve(y_test, y_proba_lstm, "LSTM - ROC Curve", "lstm_roc_curve.png")

lstm_model.save(os.path.join(MODELS_DIR, "lstm_model.keras"))
with open(os.path.join(MODELS_DIR, "tokenizer_config.json"), "w") as f:
    f.write(tokenizer.to_json())
print("Saved LSTM model and tokenizer config to", MODELS_DIR)

## 9. Model Comparison

Accuracy, precision, recall, F1-score, and ROC-AUC for both models side by side.

In [ ]:
results = pd.DataFrame([
    {
        "Model": "Logistic Regression (TF-IDF)",
        "Accuracy": accuracy_score(y_test, y_pred_lr),
        "Precision": precision_score(y_test, y_pred_lr),
        "Recall": recall_score(y_test, y_pred_lr),
        "F1-score": f1_score(y_test, y_pred_lr),
        "ROC-AUC": auc_lr,
    },
    {
        "Model": "LSTM",
        "Accuracy": accuracy_score(y_test, y_pred_lstm),
        "Precision": precision_score(y_test, y_pred_lstm),
        "Recall": recall_score(y_test, y_pred_lstm),
        "F1-score": f1_score(y_test, y_pred_lstm),
        "ROC-AUC": auc_lstm,
    },
]).set_index("Model")

display(results)

results.plot(kind="bar", figsize=(9, 5), ylim=(0, 1))
plt.title("Model Comparison: Classic ML vs Deep Learning")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "model_comparison.png"), dpi=150)
plt.show()

results.to_csv(os.path.join(OUTPUT_DIR, "model_comparison.csv"))
print("Saved comparison table to", os.path.join(OUTPUT_DIR, "model_comparison.csv"))

## 10. Discussion

**Findings.** Fill this in after running the notebook with the actual numbers from the comparison table above — e.g. which model achieved higher precision/recall, and whether the deep learning model's extra complexity translated into a meaningful accuracy gain over the TF-IDF + Logistic Regression baseline.

**Data leakage from duplicate emails.** An earlier run of this pipeline *without* the deduplication step in Section 3 produced suspiciously perfect results (accuracy/precision/recall/ROC-AUC all above 99.9% for both models). Investigating the raw CSV showed why: over half of its rows are exact duplicates of just a few hundred unique emails (some single messages — automated logs, bulk mailers — appear 1,500–4,500 times). A random train/test split without deduplication lets identical emails land on both sides of the split, so both models were partly just memorizing and recognizing duplicates rather than learning generalizable spam/phishing patterns. Deduplicating on email text before splitting (Section 3) removes this leakage, so the metrics reported above reflect genuine generalization to unseen emails.

**Cybersecurity implications.** In phishing/spam filtering, **false negatives** (a malicious email reaching the inbox) are usually more costly than **false positives** (a legitimate email flagged), since a single missed phishing email can lead to credential theft or malware execution. This makes **recall** on the phishing/spam class at least as important as overall accuracy when picking a model or a decision threshold for production use.

**Limitations.**
- Enron Spam Data labels emails as `spam` vs `ham`, not `phishing` vs `legitimate` specifically — spam is a superset that includes phishing but also bulk advertising, scams, etc. Results here should be read as "malicious/unwanted email detection" rather than a phishing-specific benchmark.
- No hyperparameter tuning (e.g. grid search over TF-IDF vocabulary size, LSTM units, learning rate) was performed — both models use reasonable defaults.
- The LSTM uses randomly initialized embeddings trained from scratch; pretrained embeddings (GloVe/Word2Vec) or a transformer (BERT) could improve results further, at the cost of training time and complexity.
- Deduplication here is exact-match on email text; near-duplicate emails (e.g. the same message with a different date/signature) are not caught and could still cause a small amount of residual leakage.

**Potential improvements.**
- Train/evaluate on a dataset with explicit phishing labels (e.g. a dedicated phishing-email corpus) in addition to Enron Spam, to validate generalization.
- Add pretrained word embeddings to the LSTM, or swap it for a fine-tuned BERT model, as the assignment brief also allows.
- Tune the classification threshold (instead of a fixed 0.5) using the ROC curve to trade off precision and recall based on the deployment's tolerance for false negatives vs false positives.